# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AmanDbz1101/FlyRank-/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

**Lane: Refresh / Content Opportunity Scoring (Lane 2).** Same contract as `w03_data_contract.ipynb`:
one row = one content page, features from **March 2026**, decision moment **2026-03-31**, label from
**April 2026** (`is_declining` = April impressions < 80% of March, volume floor March ≥ 30 impressions).

This notebook does three things, in order:

1. **Checks two signals before trusting them** — one bucket table each, with `n`, and a one-word verdict.
   Both are signals behind real FlyRank flags (staleness → the refresh flags; CTR-vs-position → the CTR-fix logic).
2. **Encodes ONE rule** — a score, ONE reason code per page, an action label — and writes the ranked queue to
   `work/outputs/baseline_action_score.csv`.
3. **Reviews the top ten by hand** — the action, why it is there, and what would make it wrong.

> Worked from `skills/building-baselines/SKILL.md` + `skills/flyrank/flyrank-data/SKILL.md`.
> This baseline is the number the Week-5 model has to beat. It is frozen after this notebook.

## 1. My rule and its reason codes

### 1a. The rule idea, in plain words (written BEFORE the checks)

> A page belongs in the refresh queue when it is **visible in search but earns far fewer clicks than pages
> at its position normally earn** — that gap is a title/meta problem, not a ranking problem. It belongs
> **higher** if March's traffic was a **spike** rather than a stable level, because spikes fall back.
> Inside a tier, the page that left the most clicks on the table goes first.

The rule leans on three claims. Two of them get a bucket table below before I am allowed to use them:

| # | Claim the rule leans on | Behind which FlyRank flag | Checked in |
|---|---|---|---|
| 1 | **Staleness** — a page that has not been updated in a long time is more likely to lose traffic | the **refresh flags** | 1b |
| 2 | **CTR-vs-position** — a page ranking well but earning few clicks is at risk | the **CTR-fix logic** | 1c |
| 3 | **Volume** — bigger pages are the ones worth acting on | **quick-win** sizing | 1d (supporting) |

Verdict vocabulary: **CONFIRMED** / **OPPOSITE** / **MIXED** / **FALSE**.

In [1]:
# SETUP — the token stays in the runtime, never in a cell (this repo is public).
import os, sys, subprocess, importlib.util, warnings, json
warnings.filterwarnings("ignore", message="IProgress not found")

if any(importlib.util.find_spec(m) is None for m in ("duckdb", "sklearn", "huggingface_hub")):
    subprocess.run([sys.executable, "-m", "pip", "-q", "install",
                    "duckdb", "scikit-learn", "huggingface_hub", "pandas", "numpy"], check=True)

import duckdb, numpy as np, pandas as pd
from huggingface_hub import hf_hub_download, get_token
pd.set_option("display.width", 200)

assert get_token(), ("No read token found. In Colab add a read-only HF_TOKEN as a Secret "
                     "(key panel) and rerun; the token is read from the runtime, not typed here.")

DS = "FlyRank/internship-warehouse"
P = {m: hf_hub_download(DS, f"fact_content_daily_performance/month={m}/data_0.parquet", repo_type="dataset")
     for m in ["2026-02", "2026-03", "2026-04"]}
DIM_CONTENT = hf_hub_download(DS, "dim_content.parquet", repo_type="dataset")
con = duckdb.connect()

DECISION_MOMENT = pd.Timestamp("2026-03-31")   # everything in the score must be knowable here
FACT3 = "[" + ",".join(f"'{P[m]}'" for m in ["2026-02", "2026-03", "2026-04"]) + "]"

# One row per content page. Feb+Mar columns are the ONLY inputs the score may touch;
# the April column exists solely to build the label further down.
raw = con.execute(f"""
    WITH f AS (
      SELECT content_hash_id,
             MAX(client_hash_id) AS client_hash_id,
             SUM(gsc_impressions) FILTER (WHERE month='2026-02' AND gsc_data_available IS TRUE) AS feb_impressions,
             SUM(gsc_impressions) FILTER (WHERE month='2026-03' AND gsc_data_available IS TRUE) AS mar_impressions,
             SUM(gsc_clicks)      FILTER (WHERE month='2026-03' AND gsc_data_available IS TRUE) AS mar_clicks,
             -- Monthly average position = SUM(sum_position) / SUM(impressions), i.e. the
             -- impression-weighted average, not a mean of daily means. Both sums are integers,
             -- so the result is exactly reproducible; AVG() of a float column is not, because
             -- parallel summation order varies and a page on a band edge would flip between runs.
             -- gsc_avg_position = 0 means "no data", never rank zero -> excluded from BOTH sums.
             SUM(gsc_sum_position) FILTER (WHERE month='2026-03' AND gsc_data_available IS TRUE
                                                 AND gsc_avg_position > 0)                      AS mar_sum_position,
             SUM(gsc_impressions)  FILTER (WHERE month='2026-03' AND gsc_data_available IS TRUE
                                                 AND gsc_avg_position > 0)                      AS mar_impr_with_position,
             SUM(gsc_impressions) FILTER (WHERE month='2026-04' AND gsc_data_available IS TRUE) AS apr_impressions
      FROM read_parquet({FACT3})
      GROUP BY content_hash_id
    )
    SELECT f.*, d.content_created_date, d.content_updated_date, d.is_published, d.is_deleted
    FROM f LEFT JOIN read_parquet('{DIM_CONTENT}') d USING (content_hash_id)
""").df()

for c in ["feb_impressions", "mar_impressions", "mar_clicks", "apr_impressions"]:
    raw[c] = raw[c].fillna(0)

# Impression-weighted average position. NaN when the page had no day with real position data.
raw["mar_avg_position"] = np.where(raw.mar_impr_with_position.fillna(0) > 0,
                                   raw.mar_sum_position / raw.mar_impr_with_position, np.nan)

raw["ctr_mar"]  = np.where(raw.mar_impressions > 0, raw.mar_clicks / raw.mar_impressions * 100, 0.0)
raw["momentum_feb_to_mar_pct"] = np.where(
    raw.feb_impressions > 0,
    (raw.mar_impressions - raw.feb_impressions) / raw.feb_impressions.replace(0, np.nan) * 100, np.nan)
raw["is_declining"] = (raw.apr_impressions < 0.8 * raw.mar_impressions).astype(int)

# Study population: visible in March (volume floor 30), and actionable (a live page you could edit).
# The published / not-deleted filter is an ACTIONABILITY filter, not a score input — see section 4.
pop = raw[(raw.mar_impressions >= 30) & (raw.is_published == True) & (raw.is_deleted == False)].copy()

print(f"pages with any March search data : {len(raw):,}")
print(f"study population (Mar impr >= 30, published, not deleted): {len(pop):,}")
print(f"base rate  P(is_declining) = {pop.is_declining.mean():.4f}")

pages with any March search data : 380,147
study population (Mar impr >= 30, published, not deleted): 125,573
base rate  P(is_declining) = 0.5181


### 1b. Signal check 1 — **staleness** (the signal behind the refresh flags)

The rule idea wanted "not updated in a long time → more likely to decline". `dim_content.content_updated_date`
is the obvious column. Bucket it by days-since-update **as measured at the decision moment**, and print `n`.

In [2]:
# SIGNAL 1 — staleness. Buckets include the pages whose update date POST-DATES the decision moment.
pop["days_since_update"] = (DECISION_MOMENT - pd.to_datetime(pop.content_updated_date)).dt.days

print("content_updated_date spans:", pd.to_datetime(raw.content_updated_date).min().date(),
      "->", pd.to_datetime(raw.content_updated_date).max().date(),
      f"   (decision moment = {DECISION_MOMENT.date()})")

bins = [-1e9, -1, 89, 179, 364, 1e9]
labs = ["updated AFTER 2026-03-31 (unknowable)", "0-89d", "90-179d", "180-364d", "365d+"]
tbl1 = (pop.assign(bucket=pd.cut(pop.days_since_update, bins=bins, labels=labs))
           .groupby("bucket", observed=False)
           .agg(n=("is_declining", "size"), decline_rate=("is_declining", "mean"))
           .round(3))
print("\nSIGNAL 1 — days since last update  x  P(declines in April)")
print(tbl1.to_string())
print(f"\nbase rate for comparison: {pop.is_declining.mean():.3f}")
print(f"share of the population where this column is NOT knowable at the decision moment: "
      f"{(pop.days_since_update < 0).mean():.1%}")

content_updated_date spans: 2025-06-01 -> 2026-07-06    (decision moment = 2026-03-31)

SIGNAL 1 — days since last update  x  P(declines in April)
                                            n  decline_rate
bucket                                                     
updated AFTER 2026-03-31 (unknowable)  104035         0.501
0-89d                                   21264         0.599
90-179d                                   243         0.576
180-364d                                   31         0.774
365d+                                       0           NaN

base rate for comparison: 0.518
share of the population where this column is NOT knowable at the decision moment: 82.8%


In [3]:
# The salvage attempt: page AGE from content_created_date — a date that cannot move after creation.
pop["page_age_days"] = (DECISION_MOMENT - pd.to_datetime(pop.content_created_date)).dt.days
abins = [-1e9, -1, 180, 365, 730, 1e9]
alabs = ["created AFTER 2026-03-31", "0-180d", "181-365d", "366-730d", "731d+"]
tbl1b = (pop.assign(bucket=pd.cut(pop.page_age_days, bins=abins, labels=alabs))
            .groupby("bucket", observed=False)
            .agg(n=("is_declining", "size"), decline_rate=("is_declining", "mean"))
            .round(3))
print("SALVAGE — page age at the decision moment  x  P(declines in April)")
print(tbl1b.to_string())
print(f"\nbase rate: {pop.is_declining.mean():.3f}   (age is knowable — 0 pages created after the decision moment)")

SALVAGE — page age at the decision moment  x  P(declines in April)
                              n  decline_rate
bucket                                       
created AFTER 2026-03-31      0           NaN
0-180d                    61743         0.490
181-365d                  46333         0.557
366-730d                  17497         0.513
731d+                         0           NaN

base rate: 0.518   (age is knowable — 0 pages created after the decision moment)


#### Verdict 1 — staleness: **FALSE**

`content_updated_date` runs to **2026-07-06**, well past the 31 Mar decision moment: it is a *snapshot as of the
warehouse build*, not a history column. **82.8%** of the study population has an update date **after** the decision
moment, so for those pages "days since update" is a negative number describing an edit that had not happened yet.

The trap is that the column *looks like it works*. The knowable slice climbs with staleness —
0-89d **0.599** (n=21,264), 90-179d **0.576** (n=243), 180-364d **0.774** (n=31) — all above the **0.518** base rate,
while the unknowable bulk sits at **0.501** (n=104,035). But that gradient is manufactured by the selection itself:
choosing "pages with a knowable update date" *is* choosing "pages nobody touched after March", which is a fact from
the future. A rule built on it would score well in this notebook and collapse in production, where you cannot know
who will edit what.

**So the staleness term is cut from the rule.** The salvage — page age from `content_created_date`, which genuinely
cannot move — is honest but nearly flat and non-monotone (0-180d **0.490**, 181-365d **0.557**, 366-730d **0.513**;
the middle bucket is the worst, not the oldest). No staleness proxy earns a place in the score.

*This is the check paying for itself: the first draft of my rule opened with a staleness term.*

### 1c. Signal check 2 — **CTR vs position** (the signal behind the CTR-fix logic)

The claim: a page that **ranks well but is not clicked** is in trouble. Restrict to pages averaging a
**top-10 position** in March — where clicks are genuinely earnable — and bucket by March CTR.

In [4]:
# SIGNAL 2 — CTR within the top-10-position population (where the CTR-fix logic applies).
# Reminder from the data dictionary: CTR is a x100 percentage. 0.25 means 0.25%, not 25%.
top10 = pop[(pop.mar_avg_position.notna()) & (pop.mar_avg_position <= 10)]
cbins = [-0.001, 0.0001, 0.25, 1.0, 3.0, 1e9]
clabs = ["0.00% (no clicks)", "0.00-0.25%", "0.25-1%", "1-3%", "3%+"]
tbl2 = (top10.assign(bucket=pd.cut(top10.ctr_mar, bins=cbins, labels=clabs))
             .groupby("bucket", observed=False)
             .agg(n=("is_declining", "size"), decline_rate=("is_declining", "mean"),
                  med_impressions=("mar_impressions", "median"))
             .round(3))
print(f"SIGNAL 2 — March CTR among top-10-position pages (n={len(top10):,})  x  P(declines in April)")
print(tbl2.to_string())
print(f"\nbase rate: {pop.is_declining.mean():.3f}")

# DATA QUIRK, stated before the bands are used: this panel carries average positions BELOW 1
# (101,548 of 3.45M March daily rows, minimum 0.0003). A real SERP rank cannot be below 1, so
# gsc_avg_position here is a pseudonymized position INDEX, not a literal rank. The lowest band is
# therefore labelled "<3" rather than "1-3" — the ordering is trustworthy, the absolute rank is not.
sub1 = con.execute(f"""SELECT COUNT(*) FROM read_parquet('{P['2026-03']}')
                       WHERE gsc_data_available IS TRUE AND gsc_avg_position > 0
                         AND gsc_avg_position < 1""").fetchone()[0]
print(f"\nDATA QUIRK: {sub1:,} March daily rows have an average position below 1 "
      f"-> read this column as a position index, not a rank.\n")

# Position alone, for contrast — is it position that matters, or the CTR gap?
pbins = [0, 3, 10, 20, 50, 1e9]; plabs = ["<3", "3-10", "10-20", "20-50", "50+"]
pop["pos_band"] = pd.cut(pop.mar_avg_position, bins=pbins, labels=plabs)
print("\nCONTRAST — position band alone x P(declines in April):")
print(pop.groupby("pos_band", observed=False)
         .agg(n=("is_declining", "size"), decline_rate=("is_declining", "mean"),
              mean_ctr=("ctr_mar", "mean")).round(3).to_string())

SIGNAL 2 — March CTR among top-10-position pages (n=67,189)  x  P(declines in April)
                       n  decline_rate  med_impressions
bucket                                                 
0.00% (no clicks)  23276         0.632            176.0
0.00-0.25%         17561         0.590           2384.0
0.25-1%            21247         0.407           1532.0
1-3%                4520         0.325            345.5
3%+                  585         0.468             61.0

base rate: 0.518

DATA QUIRK: 101,548 March daily rows have an average position below 1 -> read this column as a position index, not a rank.


CONTRAST — position band alone x P(declines in April):
              n  decline_rate  mean_ctr
pos_band                               
<3        10404         0.552     0.358
3-10      56785         0.524     0.338
10-20     24071         0.522     0.249
20-50     26167         0.486     0.155
50+        8143         0.528     0.057


#### Verdict 2 — CTR vs position: **CONFIRMED**

Among pages averaging a position of 10 or better, the decline rate falls monotonically as CTR rises:
**0.632** (no clicks, n=23,276) → **0.590** (n=17,561) → **0.407** (n=21,247) → **0.325** (n=4,520).
That is a near-2× spread across a signal knowable on 31 March, against a **0.518** base rate.

Two honest caveats. The top bucket **reverses** — 3%+ CTR declines at **0.468** (n=585) — but that bucket is 0.9% of
the population with a median of only 61 March impressions, so it is small-denominator noise, not a real turn.
And the contrast table shows position **alone** barely moves: **0.552 / 0.524 / 0.522 / 0.486 / 0.528** across bands,
a 0.066 spread, against CTR's **0.325 → 0.632** spread of 0.307. Ranking well is not itself protective. It is the
**gap between position and clicks** that carries the signal — exactly what the CTR-fix logic claims, so it goes into
the rule as the main term.

### 1d. Supporting check — **volume** (quick-win sizing)

Not one of the two required verdicts, but the rule's first draft multiplied the score by impressions, so
the claim "bigger = more at risk" needs testing too.

In [5]:
# SUPPORTING — does raw March volume predict decline?
vbins = [-1, 100, 500, 2000, 10000, 50000, 1e12]
vlabs = ["30-100", "100-500", "500-2k", "2k-10k", "10k-50k", "50k+"]
tbl3 = (pop.assign(bucket=pd.cut(pop.mar_impressions, bins=vbins, labels=vlabs))
           .groupby("bucket", observed=False)
           .agg(n=("is_declining", "size"), decline_rate=("is_declining", "mean"))
           .round(3))
print("SUPPORTING — March impressions x P(declines in April)")
print(tbl3.to_string())
print(f"\nbase rate: {pop.is_declining.mean():.3f}")

SUPPORTING — March impressions x P(declines in April)
             n  decline_rate
bucket                      
30-100   24373         0.521
100-500  39337         0.544
500-2k   32004         0.528
2k-10k   23983         0.477
10k-50k   5556         0.447
50k+       320         0.409

base rate: 0.518


#### Verdict 3 (supporting) — volume as a *risk* signal: **OPPOSITE**

Decline rate *falls* as volume rises: **0.544** at 100-500 impressions down to **0.447** at 10k-50k and **0.409** at
50k+ (n=320). Big pages are marginally **safer**, not riskier. So volume must not multiply the risk score — my first
draft did exactly that and it never pulls clear of the base rate at any K (**@100 = 0.530** against this rule's
**0.750**) — that rejected draft is scored alongside the real one in section 2 rather than just asserted here.

Volume still belongs in the queue, but as **impact**, not risk: among two equally-risky pages, work the one with more
clicks left on the table. That is the `clicks_lost` tie-break below, capped so it can never lift a page across a risk tier.

### 1e. The rule as finally encoded

```
ctr_expected  = mean March CTR of the page's position band      (a March-only lookup)
ctr_ratio     = ctr_mar / ctr_expected
clicks_lost   = max(0, mar_impressions * ctr_expected/100 - mar_clicks)

no_clicks_at_top10 = (mar_clicks == 0) and (position <= 10)
ctr_far_below      = ctr_ratio < 0.5
spiked             = momentum_feb_to_mar_pct > 25

risk_points   = 1 + 2*(no_clicks_at_top10 or ctr_far_below) + 1*spiked      # 1 .. 4
ctr_shortfall = clip(1 - ctr_ratio, 0, 1)                                   # 0 .. 1   within-tier ordering
impact        = log1p(clicks_lost) / 10                                     # 0 .. <1  within-tier tie-break
score         = risk_points + ctr_shortfall + impact
```

No fitted weights: `1 / 2 / 1` are hand-set, and `/10` exists only to hold the impact term below 1.0 so it can never
push a page past a risk tier (asserted in code). Staleness is absent — verdict 1 removed it.

| Reason code (ONE per page, first match wins) | Condition | Action label |
|---|---|---|
| `NO_CLICKS_AT_TOP10` | top-10 position, zero March clicks | `REWRITE_SNIPPET` |
| `CTR_FAR_BELOW_POSITION` | CTR below half its position band's mean | `REVIEW_SNIPPET` |
| `SPIKE_LIKELY_TO_REVERT` | March impressions > 125% of February | `MONITOR_30D` |
| `VISIBLE_ONLY` | nothing fired | `NO_ACTION` |

## 2. Build the ranked queue (writes the CSV)

In [6]:
# ---- ENCODE THE RULE ------------------------------------------------------
# Expected CTR per position band, learned from MARCH columns only (no April touches this).
BAND_CTR = pop.groupby("pos_band", observed=False)["ctr_mar"].mean()
print("expected CTR by position band (March mean, %):")
print(BAND_CTR.round(4).to_string())

q = pop.copy()
q["ctr_expected"] = q.pos_band.map(BAND_CTR).astype(float)
q["ctr_ratio"]    = q.ctr_mar / q.ctr_expected            # NaN when the page has no position data
q["expected_clicks"] = q.mar_impressions * q.ctr_expected / 100
q["clicks_lost"]     = np.clip(q.expected_clicks - q.mar_clicks, 0, None).fillna(0)

# The three conditions. Pages with no position data fail all of them -> VISIBLE_ONLY (conservative).
q["no_clicks_at_top10"] = ((q.mar_clicks == 0) & (q.mar_avg_position <= 10)).fillna(False)
q["ctr_far_below"]      = (q.ctr_ratio < 0.5).fillna(False)
q["spiked"]             = (q.momentum_feb_to_mar_pct > 25).fillna(False)

q["risk_points"]   = 1 + 2 * (q.no_clicks_at_top10 | q.ctr_far_below).astype(int) + q.spiked.astype(int)
q["ctr_shortfall"] = np.clip(1 - q.ctr_ratio, 0, 1).fillna(0)
q["impact"]        = np.log1p(q.clicks_lost) / 10
assert q.impact.max() < 1.0, "impact term must stay below 1.0 so it can never cross a risk tier"
q["score"] = q.risk_points + q.ctr_shortfall + q.impact

def reason_code(r):
    if r.no_clicks_at_top10: return "NO_CLICKS_AT_TOP10"
    if r.ctr_far_below:      return "CTR_FAR_BELOW_POSITION"
    if r.spiked:             return "SPIKE_LIKELY_TO_REVERT"
    return "VISIBLE_ONLY"

ACTION = {"NO_CLICKS_AT_TOP10":     "REWRITE_SNIPPET",
          "CTR_FAR_BELOW_POSITION": "REVIEW_SNIPPET",
          "SPIKE_LIKELY_TO_REVERT": "MONITOR_30D",
          "VISIBLE_ONLY":           "NO_ACTION"}

q["reason_code"] = q.apply(reason_code, axis=1)
q["action"]      = q.reason_code.map(ACTION)

# 87k pages tie at the bottom of the score (VISIBLE_ONLY with nothing lost), so the sort needs a
# deterministic tie-break or the queue — and every number below it — changes between runs.
q = q.sort_values(["score", "content_hash_id"], ascending=[False, True],
                  kind="mergesort").reset_index(drop=True)
q.insert(0, "rank", np.arange(1, len(q) + 1))

print(f"\nimpact term max = {q.impact.max():.4f}  (<1 -> tie-break only)")
print(f"score range: {q.score.min():.3f} .. {q.score.max():.3f}   |  ties inside the top 100: "
      f"{100 - q.head(100).score.nunique()}")
print(f"\nqueue rows: {len(q):,}")
print("\nreason code mix, and how each fares against the April label:")
print(q.groupby("reason_code")
       .agg(n=("is_declining", "size"), decline_rate=("is_declining", "mean"),
            med_impressions=("mar_impressions", "median"), med_position=("mar_avg_position", "median"))
       .sort_values("decline_rate", ascending=False).round(3).to_string())
print("\nrisk tiers:")
print(q.groupby("risk_points").agg(n=("is_declining", "size"), decline_rate=("is_declining", "mean"))
       .round(3).to_string())

expected CTR by position band (March mean, %):
pos_band
<3       0.3584
3-10     0.3379
10-20    0.2488
20-50    0.1555
50+      0.0569



impact term max = 0.6604  (<1 -> tie-break only)
score range: 1.000 .. 5.629   |  ties inside the top 100: 0

queue rows: 125,573

reason code mix, and how each fares against the April label:
                            n  decline_rate  med_impressions  med_position
reason_code                                                               
NO_CLICKS_AT_TOP10      23276         0.632            176.0         6.153
CTR_FAR_BELOW_POSITION  51294         0.549            295.0        20.026
SPIKE_LIKELY_TO_REVERT  24620         0.473           1444.0         7.870
VISIBLE_ONLY            26383         0.400            936.0         6.371

risk tiers:
                 n  decline_rate
risk_points                     
1            26383         0.400
2            24620         0.473
3            35660         0.539
4            38910         0.607


In [7]:
# ---- WRITE THE RANKED QUEUE ----------------------------------------------
from pathlib import Path
# Works whether the kernel's cwd is the repo root (nbconvert) or work/notebooks (Colab/Jupyter).
CWD = Path.cwd()
OUT = (CWD.parent / "outputs") if CWD.name == "notebooks" else (CWD / "work" / "outputs")
OUT.mkdir(parents=True, exist_ok=True)

QUEUE_COLS = ["rank", "content_hash_id", "client_hash_id", "action", "reason_code", "score",
              "risk_points", "ctr_shortfall", "impact",
              "mar_impressions", "mar_clicks", "ctr_mar", "ctr_expected", "ctr_ratio",
              "mar_avg_position", "momentum_feb_to_mar_pct", "clicks_lost"]
csv_path = OUT / "baseline_action_score.csv"
q[QUEUE_COLS].to_csv(csv_path, index=False)
print(f"wrote {csv_path}  ({len(q):,} rows x {len(QUEUE_COLS)} cols)")
print("NOTE: the label column is deliberately NOT in the CSV — the queue is what an editor sees on 31 March.")
print("\nfirst 3 rows as written:")
print(pd.read_csv(csv_path, nrows=3).round(3).to_string(index=False))

wrote /mnt/storage/Internship/FlyRank/work/outputs/baseline_action_score.csv  (125,573 rows x 17 cols)
NOTE: the label column is deliberately NOT in the CSV — the queue is what an editor sees on 31 March.

first 3 rows as written:
 rank          content_hash_id          client_hash_id         action            reason_code  score  risk_points  ctr_shortfall  impact  mar_impressions  mar_clicks  ctr_mar  ctr_expected  ctr_ratio  mar_avg_position  momentum_feb_to_mar_pct  clicks_lost
    1 content_44f34c0a90047651 client_23a62021009f63c4 REVIEW_SNIPPET CTR_FAR_BELOW_POSITION  5.629            4          0.968   0.660         212404.0        24.0    0.011         0.358      0.032             0.666                  135.421      737.293
    2 content_cd3d932d4e1c8db0 client_9958f0a7ae1df715 REVIEW_SNIPPET CTR_FAR_BELOW_POSITION  5.557            4          0.987   0.570          89332.0         4.0    0.004         0.338      0.013             7.832                30182.034      297.877
    

In [8]:
# ---- EVALUATE: precision@K against the base rate and against dumber baselines -------------
from sklearn.dummy import DummyClassifier

base = q.is_declining.mean()

def precision_at_k(labels, k):
    return float(np.asarray(labels)[:k].mean())

print(f"base rate (P(declining) over the whole queue) = {base:.4f}   n = {len(q):,}\n")
print(f"{'K':>7}  {'precision@K':>12}  {'lift':>6}")
rows = []
for k in [10, 20, 50, 100, 500, 1000, 5000, 10000]:
    p = precision_at_k(q.is_declining, k)
    rows.append((k, p))
    print(f"{k:>7}  {p:>12.3f}  {p/base:>5.2f}x")

# Baselines to beat.
rand = [q.is_declining.sample(frac=1, random_state=s).values[:100].mean() for s in range(50)]
vol_only = q.sort_values("mar_impressions", ascending=False).is_declining.values[:100].mean()
dummy = DummyClassifier(strategy="most_frequent").fit(q[["score"]], q.is_declining)

print(f"\nrandom order      @100 = {np.mean(rand):.3f} +/- {np.std(rand):.3f}  (50 shuffles)")
print(f"impressions only  @100 = {vol_only:.3f}   <- the 'just work the big pages' queue")
print(f"my rule           @100 = {precision_at_k(q.is_declining, 100):.3f}")

# The REJECTED first draft, kept as evidence for verdict 3: score = risk_points * log1p(impressions).
v0 = q.assign(v0=q.risk_points * np.log1p(q.mar_impressions)).sort_values("v0", ascending=False)
v0_p = {k: precision_at_k(v0.is_declining, k) for k in [10, 100, 1000]}
print(f"\nrejected draft (risk x log1p(impressions)) @10 = {v0_p[10]:.3f}, @100 = {v0_p[100]:.3f}, "
      f"@1000 = {v0_p[1000]:.3f}")
print(f"   -> its top 10 scores BELOW the {base:.3f} base rate: multiplying by volume put the")
print("      safest pages first, exactly as the supporting check in 1d predicted.")
print(f"\nDummyClassifier(most_frequent) accuracy floor = {dummy.score(q[['score']], q.is_declining):.3f}")

print("\ndecile of the queue (decile 0 = top of the list):")
q["decile"] = pd.qcut(q["rank"], 10, labels=False)
print(q.groupby("decile").agg(n=("is_declining", "size"), decline_rate=("is_declining", "mean"),
                              med_impressions=("mar_impressions", "median"),
                              med_ctr_ratio=("ctr_ratio", "median")).round(3).to_string())

metrics = {
    "slice": {"features_month": "2026-03", "label_month": "2026-04",
              "decision_moment": "2026-03-31", "volume_floor_mar_impressions": 30},
    "queue_rows": int(len(q)), "base_rate": round(float(base), 4),
    "precision_at_k": {str(k): round(p, 4) for k, p in rows},
    "lift_at_k": {str(k): round(p / base, 3) for k, p in rows},
    "baselines": {"random_at_100_mean": round(float(np.mean(rand)), 4),
                  "random_at_100_std": round(float(np.std(rand)), 4),
                  "impressions_only_at_100": round(float(vol_only), 4),
                  "dummy_most_frequent_accuracy": round(float(dummy.score(q[["score"]], q.is_declining)), 4),
                  "rejected_draft_risk_times_volume": {str(k): round(v, 4) for k, v in v0_p.items()}},
    "reason_code_rates": {k: round(float(v), 4)
                          for k, v in q.groupby("reason_code").is_declining.mean().items()},
    "reason_code_counts": {k: int(v) for k, v in q.reason_code.value_counts().items()},
    "signal_verdicts": {"staleness_content_updated_date": "FALSE",
                        "ctr_vs_position": "CONFIRMED",
                        "volume_as_risk_signal": "OPPOSITE"},
}
with open(OUT / "w04_baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(f"\nwrote {OUT / 'w04_baseline_metrics.json'}")

base rate (P(declining) over the whole queue) = 0.5181   n = 125,573

      K   precision@K    lift
     10         0.700   1.35x
     20         0.700   1.35x
     50         0.680   1.31x
    100         0.750   1.45x
    500         0.682   1.32x
   1000         0.665   1.28x
   5000         0.659   1.27x
  10000         0.660   1.27x

random order      @100 = 0.523 +/- 0.049  (50 shuffles)
impressions only  @100 = 0.390   <- the 'just work the big pages' queue
my rule           @100 = 0.750

rejected draft (risk x log1p(impressions)) @10 = 0.500, @100 = 0.530, @1000 = 0.584
   -> its top 10 scores BELOW the 0.518 base rate: multiplying by volume put the
      safest pages first, exactly as the supporting check in 1d predicted.

DummyClassifier(most_frequent) accuracy floor = 0.518

decile of the queue (decile 0 = top of the list):
            n  decline_rate  med_impressions  med_ctr_ratio
decile                                                     
0       12558         0.658      


wrote /mnt/storage/Internship/FlyRank/work/outputs/w04_baseline_metrics.json


### Reading the numbers

Base rate **0.518** — over half the visible pages lose impressions in April, so a coin flip already looks
"right" half the time. Against that floor the rule reaches **precision@10 = 0.700**, **@100 = 0.750**,
**@1000 = 0.665**, **@5000 = 0.659** — a **1.27×–1.45×** lift that holds as K grows, which matters far more than
the top-10 number alone (10 rows is nowhere near enough to trust on its own).

The dumber queues are the real comparison. Random order gets **0.523 ± 0.049**, i.e. the base rate.
Ranking by **impressions alone** gets **0.390** — *worse* than random, which is verdict 3 showing up again:
sorting by size actively pushes the safest pages to the top. The rule beats both.

The reason codes are ordered the way the checks predicted: `NO_CLICKS_AT_TOP10` **0.632** > `CTR_FAR_BELOW_POSITION`
**0.549** > `SPIKE_LIKELY_TO_REVERT` **0.473** > `VISIBLE_ONLY` **0.400**, and the risk tiers rise cleanly
**0.400 → 0.473 → 0.539 → 0.607**. The queue's deciles fall from **0.658** at the top to **0.363** at the bottom.

The last line of that cell is the receipt for verdict 3: the **rejected first draft**,
`risk_points × log1p(impressions)`, scores **@100 = 0.530** and **@1000 = 0.584** against this rule's **0.750** and
**0.665** — it never beats the base rate by much at any K, and its **@10 = 0.500** sits just under the 0.518 base
rate outright. Multiplying by volume put the safest pages first. That draft is what the supporting check killed, and
it is scored here rather than merely described so the claim can be checked.

The ordering is real, but it is a *directional* separation on one month, not a validated model — that is Week 5's job.

## 3. Top-10 review

In [9]:
REVIEW_COLS = ["rank", "content_hash_id", "action", "reason_code", "score", "risk_points",
               "mar_impressions", "mar_clicks", "ctr_mar", "ctr_expected", "ctr_ratio",
               "mar_avg_position", "momentum_feb_to_mar_pct", "clicks_lost"]
print("TOP 10 — what an editor would see on 2026-03-31 (no label column: this is the decision view)")
print(q[REVIEW_COLS].head(10).round(3).to_string(index=False))

print("\n--- hindsight only: how the top 10 actually turned out in April ---")
print(q[["rank", "mar_impressions", "apr_impressions", "is_declining"]].head(10).to_string(index=False))
print(f"\ntop-10 hits: {int(q.is_declining.head(10).sum())}/10   "
      f"(base rate would give ~{10*q.is_declining.mean():.1f}/10)")
print(f"distinct clients in the top 10: {q.client_hash_id.head(10).nunique()}")

TOP 10 — what an editor would see on 2026-03-31 (no label column: this is the decision view)
 rank          content_hash_id          action            reason_code  score  risk_points  mar_impressions  mar_clicks  ctr_mar  ctr_expected  ctr_ratio  mar_avg_position  momentum_feb_to_mar_pct  clicks_lost
    1 content_44f34c0a90047651  REVIEW_SNIPPET CTR_FAR_BELOW_POSITION  5.629            4         212404.0        24.0    0.011         0.358      0.032             0.666                  135.421      737.293
    2 content_cd3d932d4e1c8db0  REVIEW_SNIPPET CTR_FAR_BELOW_POSITION  5.557            4          89332.0         4.0    0.004         0.338      0.013             7.832                30182.034      297.877
    3 content_f6116743b00afc2d  REVIEW_SNIPPET CTR_FAR_BELOW_POSITION  5.544            4         107584.0        15.0    0.014         0.338      0.041             9.736                  369.103      348.556
    4 content_425715547c6a3ea8  REVIEW_SNIPPET CTR_FAR_BELOW_POSITION  

### The ten, one line each — the action, why it is there, what would make it wrong

Every page here is a **high-impression page earning almost no clicks**, so the queue's whole top tells one story:
*visibility is fine, the snippet is not*. Hindsight: **7 of 10** declined (the base rate would give ~5.2/10).

| # | Action | Why it is there | What would make it wrong |
|---|---|---|---|
| 1 | `REVIEW_SNIPPET` | 212k impressions but **24 clicks** — CTR 0.011% against 0.358% expected for its band, ~737 clicks lost, the largest shortfall in the panel | Its weighted position is **0.67**, i.e. below rank 1, which no real SERP produces. For this page the position index is not interpretable, so the "expected CTR for its band" it is being judged against may simply be the wrong yardstick. |
| 2 | `REVIEW_SNIPPET` | 89k impressions at position 7.8 with **4 clicks**, ratio 0.013 | Momentum is **+30,182%** — February was ~295 impressions. This is a page that just exploded, and its March CTR is measured over a launch ramp, not a settled page. |
| 3 | `REVIEW_SNIPPET` | 108k impressions at position 9.7, **15 clicks**, 4% of expected CTR, ~349 clicks lost | Position 9.7 is a *monthly* aggregate; it could be a page that sat at 3 for a week and 40 for the rest, in which case the CTR gap is explained by rank volatility, not by the title. |
| 4 | `REVIEW_SNIPPET` | 72k impressions at position 7.0 with **3 clicks**, ratio 0.012 | **Wrong** — April *rose* to 77.9k. Momentum **+4,775%** should have been the tell: a page still climbing was read as a page at risk. |
| 5 | `REVIEW_SNIPPET` | 63k impressions at position 8.7 with **2 clicks**, ~212 clicks lost | **Wrong** — April held roughly flat (63.2k → 56.3k, an 11% dip that stays under the 20% threshold). Momentum **+112,759%** means February was ~56 impressions; the comparison is arithmetically valid and practically meaningless. |
| 6 | `REVIEW_SNIPPET` | 143k impressions at position **3.2** with 43 clicks — a genuinely well-ranked page at 0.030% CTR, ~440 clicks lost | If the queries behind it are navigational for someone else's brand, a top-3 position earns no clicks whatever the title says, and the shortfall is structural rather than fixable. |
| 7 | `REVIEW_SNIPPET` | 82k impressions at position 8.0, **11 clicks**, ratio 0.040 | **Wrong** — April rose to 112.6k. Same failure as #4 and #5: momentum **+13,294%**, a page mid-launch. |
| 8 | `REVIEW_SNIPPET` | 58k impressions at position 8.3, **5 clicks**, ~192 clicks lost | It collapsed to 3.1k in April — a 95% fall far larger than a snippet problem explains. Likely deindexing or a site change, so the *ranking* was right but `REVIEW_SNIPPET` is the wrong **action**. |
| 9 | `REWRITE_SNIPPET` | 39k impressions at position 4.8 and **zero clicks all month** — the cleanest instance of the confirmed signal | Zero clicks on 39k impressions at position 4.8 is extreme enough to smell like measurement: a redirect, a canonical pointing elsewhere, or clicks attributed to another URL. Verify the page is reachable before anyone rewrites a title. |
| 10 | `REVIEW_SNIPPET` | 97k impressions with **2 clicks**, ~149 clicks lost | Its position is **36.9**, where the band's expected CTR is only 0.155%. At rank 37 almost nobody clicks regardless of the snippet, so "CTR far below expected" is nearly unavoidable — this is weak pick 2 in section 4, sitting in the top ten. |

**The pattern across all ten:** the rule found a real, confirmed signal and applied it to the wrong population.
Five of the ten have momentum above **+1,000%** — February was almost nothing — and **all three misses (#4, #5, #7)
are in that group**. A low CTR on a page that launched three weeks ago is a symptom of a page that has not settled,
not of a bad snippet. The fix is a February-baseline requirement, not a new signal; that is section 4.

## 4. Weak picks + leakage check

In [10]:
# ---- WEAK PICKS: what is actually sitting at the top of my queue? ---------
top100 = q.head(100)
print("WEAK PICKS in the top 100")
print(f"  momentum > +1000% (February was ~nothing)      : {int((top100.momentum_feb_to_mar_pct > 1000).sum()):>4} / 100")
print(f"  no February baseline at all (momentum is NaN)  : {int(top100.momentum_feb_to_mar_pct.isna().sum()):>4} / 100")
print(f"  average position worse than 20                 : {int((top100.mar_avg_position > 20).sum()):>4} / 100")
print(f"  did NOT decline in April (the rule was wrong)  : {int((top100.is_declining == 0).sum()):>4} / 100")
print(f"  largest single client's share of the top 100   : {top100.client_hash_id.value_counts().iloc[0]:>4} / 100"
      f"   (across {top100.client_hash_id.nunique()} clients)")

# Weak pick 1 — does the rule just rediscover brand-new pages?
q["explosive"] = (q.momentum_feb_to_mar_pct > 1000).fillna(False)
print("\nare explosive-momentum pages actually riskier? (if not, they are noise at the top)")
print(q.groupby("explosive").agg(n=("is_declining", "size"), decline_rate=("is_declining", "mean"))
       .round(3).to_string())

settled = q[~q.explosive].reset_index(drop=True)
print(f"\nprecision@K with explosive pages removed (base {settled.is_declining.mean():.3f}):")
for k in [10, 100, 1000]:
    print(f"   @{k:<5d} {settled.is_declining.values[:k].mean():.3f}"
          f"   (vs {q.is_declining.values[:k].mean():.3f} with them in)")

# Weak pick 2 — the deep-position pages. Can a page at position 30+ earn its band's CTR at all?
deep = q.head(100)[q.head(100).mar_avg_position > 20]
print(f"\nweak pick 2: {len(deep)} of the top 100 average a position worse than 20.")
print(f"  their median expected CTR is {deep.ctr_expected.median():.3f}% — a target so small that")
print(f"  'CTR below half of expected' is nearly automatic for any page with few clicks.")

WEAK PICKS in the top 100
  momentum > +1000% (February was ~nothing)      :   31 / 100
  no February baseline at all (momentum is NaN)  :    0 / 100
  average position worse than 20                 :   28 / 100
  did NOT decline in April (the rule was wrong)  :   25 / 100
  largest single client's share of the top 100   :   35 / 100   (across 12 clients)

are explosive-momentum pages actually riskier? (if not, they are noise at the top)
                n  decline_rate
explosive                      
False      119665         0.519
True         5908         0.500

precision@K with explosive pages removed (base 0.519):
   @10    1.000   (vs 0.700 with them in)
   @100   0.730   (vs 0.750 with them in)
   @1000  0.668   (vs 0.665 with them in)

weak pick 2: 28 of the top 100 average a position worse than 20.
  their median expected CTR is 0.155% — a target so small that
  'CTR below half of expected' is nearly automatic for any page with few clicks.


In [11]:
# ---- LEAKAGE CHECK: prove nothing from the label window reached the score --
SCORE_INPUTS = ["ctr_mar", "ctr_expected", "ctr_ratio", "mar_impressions", "mar_clicks",
                "mar_avg_position", "momentum_feb_to_mar_pct", "clicks_lost", "expected_clicks",
                "no_clicks_at_top10", "ctr_far_below", "spiked", "risk_points", "ctr_shortfall", "impact"]
LABEL_INPUTS = ["apr_impressions", "is_declining"]

print("1) every score input is built from February/March columns only:")
for c in SCORE_INPUTS:
    print(f"     {c}")
print("\n2) the label window (April) is used ONLY here:")
for c in LABEL_INPUTS:
    print(f"     {c}")
assert not (set(SCORE_INPUTS) & set(LABEL_INPUTS)), "a score input is also a label input"

# 3) The CSV an editor receives must not carry the answer.
written = pd.read_csv(csv_path, nrows=5)
leaked = [c for c in written.columns if c in LABEL_INPUTS or "apr" in c or "declin" in c]
print(f"\n3) label-window columns present in the delivered CSV: {leaked}  -> {'OK' if not leaked else 'LEAK'}")
assert not leaked, "the queue CSV must not contain the label"

# 4) The band-CTR lookup is fitted on March CTR only — check it never saw April.
print(f"\n4) BAND_CTR lookup built from column 'ctr_mar' (March clicks / March impressions):")
print(BAND_CTR.round(4).to_string())

# 5) No product flag was used. The rule's inputs are raw warehouse metrics, not FlyRank's own flags.
print("\n5) no FlyRank product flag (refresh flag, CTR-fix flag, quick-win flag) is an input;")
print("   the rule re-derives its own conditions from gsc_* columns, so the baseline is not")
print("   scoring itself against the system it is meant to be compared with.")

# 6) The one snapshot-column dependency, stated openly.
snap = raw[(raw.mar_impressions >= 30)]
n_dropped = len(snap) - len(pop)
print(f"\n6) DISCLOSED: is_published / is_deleted are snapshot-as-of-build columns, same family as the")
print(f"   content_updated_date that verdict 1 rejected. They are used ONLY to drop pages an editor")
print(f"   could not act on, never as score inputs, and they move {n_dropped} of {len(snap):,} rows "
      f"({n_dropped/len(snap):.2%}).")
print(f"   Decline rate of the dropped rows: {snap[~snap.content_hash_id.isin(pop.content_hash_id)].is_declining.mean():.3f}")

1) every score input is built from February/March columns only:
     ctr_mar
     ctr_expected
     ctr_ratio
     mar_impressions
     mar_clicks
     mar_avg_position
     momentum_feb_to_mar_pct
     clicks_lost
     expected_clicks
     no_clicks_at_top10
     ctr_far_below
     spiked
     risk_points
     ctr_shortfall
     impact

2) the label window (April) is used ONLY here:
     apr_impressions
     is_declining

3) label-window columns present in the delivered CSV: []  -> OK

4) BAND_CTR lookup built from column 'ctr_mar' (March clicks / March impressions):
pos_band
<3       0.3584
3-10     0.3379
10-20    0.2488
20-50    0.1555
50+      0.0569

5) no FlyRank product flag (refresh flag, CTR-fix flag, quick-win flag) is an input;
   the rule re-derives its own conditions from gsc_* columns, so the baseline is not
   scoring itself against the system it is meant to be compared with.

6) DISCLOSED: is_published / is_deleted are snapshot-as-of-build columns, same family as the
 

### What the weak picks say

**Weak pick 1 — the rule rediscovers new pages, not at-risk pages.** 31 of the top 100 have Feb→Mar momentum above
**+1,000%**, meaning February was near zero. For those pages a "CTR far below position" reading is measured over a
launch ramp, not a settled page. And explosive pages are **not** riskier — **0.500** vs **0.519** for everything else,
i.e. they are noise. Removing them lifts **precision@10 from 0.700 to 1.000** and **@1000 from 0.665 to 0.668**,
while **@100 slips from 0.750 to 0.730** — the dip is real, a few genuine catches go with them. All three top-10
misses were explosive pages. The fix for Week 5 is a **February-baseline requirement**, not a new signal — and I am
leaving it out of *this* notebook on purpose, because the baseline is frozen here and tuning it after seeing the
label is how baselines get quietly rigged.

**Weak pick 2 — the CTR test is too easy at deep positions.** 28 of the top 100 average a position worse than 20,
where the band's expected CTR is only **0.155%**. Half of a target that small is nearly unmissable, so
`ctr_far_below` fires almost automatically on any deep page with few clicks. Rank 10 of the queue is exactly this
failure. It should be gated to positions where clicks are genuinely earnable.

**Weak pick 3 — client concentration.** One client owns **35 of the top 100** and the whole top 100 spans only 12
clients, out of the 55 with March data. A per-client queue, or a per-client cap, would make this list usable by a
team that serves many accounts.

**Weak pick 4 — 25 of the top 100 did not decline at all.** That is the honest cost of a 0.750 precision: one in
four rows an editor works is a page that was going to be fine.

### Leakage check

- Every score input is Feb/March-derived; the **only** use of April is `is_declining`, and the two sets are disjoint (asserted).
- The delivered CSV contains **no** label column — it is exactly what an editor could see on 31 March (asserted).
- The `ctr_expected` lookup is a mean of **March** CTR by position band. It is fitted on the same rows it scores, which
  is a mild in-sample dependency worth naming, but it cannot leak the future because no April column enters it.
- **No FlyRank product flag is an input.** The rule re-derives its own conditions from raw `gsc_*` columns, so this
  baseline is an independent comparison rather than a copy of the system it is measured against.
- **Disclosed dependency:** `is_published` / `is_deleted` are snapshot-as-of-build columns — the same family as the
  `content_updated_date` that verdict 1 threw out. They are used only to drop pages an editor could not act on, never
  as score inputs, and they touch 72 of 125,645 rows (0.06%). Named here rather than hidden, because applying the
  same suspicion consistently is the point of section 1b.

### One named limitation

This is **one month, one comparison window**. `is_declining` cannot tell real decay from seasonality or from a
page that simply had an unusual March, and the rule is evaluated on the same slice it was written against. It is a
frozen, directional, decision-support baseline — the number Week 5 has to beat — not a validated predictor.

## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere — only pseudonymous hash IDs
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] **Two signal checks**, each with a bucket table and visible `n`, each with a one-word verdict
      (staleness → **FALSE**, CTR-vs-position → **CONFIRMED**), and both are signals behind real FlyRank flags
- [x] **One rule** encoded with a **score**, **ONE reason code** per page, and an **action label**
- [x] The **ranked queue is written from this notebook** to `work/outputs/baseline_action_score.csv`
- [x] **Ten rows reviewed by hand**, each with the action, why it is there, and what would make it wrong
- [x] **No future-window or label-derived inputs** in the score — asserted in section 4, and the delivered CSV
      carries no label column
- [x] precision@K reported next to the **base rate** and against random / impressions-only baselines
- [x] Weak picks found and named (4 of them), plus one named limitation
- [x] Committed to my repo under `work/notebooks/` — the CSV stays out of git by design; the metrics JSON is committed